# 向量数据库（Vector Database）与近似最近邻搜索（ANN, Approximate Nearest Neighbor）算法
如果说 Embedding 是把文本变成了坐标，那么向量数据库就是能在上亿个高维坐标中，以毫秒级速度帮你找到“距离最近的 K 个坐标”的超级引擎。

## 向量检索底层原理

1. 彻底搞懂向量索引算法（HNSW 与 IVF）：为什么暴力搜索（Flat）会随着数据量暴增而崩溃？HNSW（可扩展小世界图）是如何做到 $\mathcal{O}(\log N)$ 检索效率的？
2. 轻量级向量数据库实战：用工业界轻量级标杆 ChromaDB，结合元数据过滤（Metadata Filtering）与 CRUD 操作，搭建本地向量存储与检索流水线。

#### 从暴力搜索到 HNSW 的算法跃迁
在拥有 $N$ 个向量的数据库中搜索最相似的 $K$ 个向量，有两类完全不同的解法：
1. 暴力搜索 (Exact Search / Flat)
    * 原理：拿 User Query 的向量，跟数据库里存储的 所有 $N$ 个向量 依次计算余弦相似度，最后取 Top-K。
    * 时间复杂度：$\mathcal{O}(N \cdot D)$（其中 $N$ 是向量数，$D$ 是向量维度）。
    * 痛点：当数据量达到数十万甚至上百万时，单次检索需要几秒甚至更久，生产环境完全不可用。

2. 近似最近邻搜索 (ANN, Approximate Nearest Neighbor)

    为了解决效率问题，工业界放弃了“100% 绝对精确”，转而牺牲极微小的精度（如 98% 召回率），换取百倍的检索速度。核心算法分为两派：
   1. IVF (Inverted File Index, 倒排文件索引)
      * 思想：聚类（Clustering）。
      * 逻辑：先把全量向量用 $K$-Means 聚成 $N$ 个桶（Cluster）。检索时，先找到 Query 落在哪个/哪几个桶里，然后只计算这几个桶内部的向量。搜索范围骤降为原来的 $1/N$。

    2. HNSW (Hierarchical Navigable Small World, 分层可导航小世界图) —— 目前最强索引
       * 思想：跳表（Skip-List）+ 小世界图（Small World Graph）。
       * 逻辑：
            * 底层（Layer 0）：一个包含所有向量节点的无向图，每个节点与其邻居建立连接。
            * 顶层（Layer N）：节点稀疏，边长极长（类似于高速公路）。
            * 检索过程：从最高层进入，利用“高速公路”快速定位到 Query 附近的大致区域；然后逐层向下跳转，直到在 Layer 0 进行局部精细搜索。
       * 优势：检索速度极快（$\mathcal{O}(\log N)$），对高维数据的表达能力远超传统的基于树（KD-Tree）或基于聚类（IVF）的算法。

#### ChromaDB 本地数据库管理与混合过滤
在实际项目中，纯开源轻量级首选是 ChromaDB 或 FAISS。今天我们使用 Python 原生且零配置的 ChromaDB，展示完整的文本向量入库、元数据附加与高级条件过滤。
> 安装依赖：`pip install chromadb`


In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# --- 1. 初始化 ChromaDB (本地持久化模式) ---
# 会在当前目录下创建一个 ./chroma_db 文件夹保存数据
client = chromadb.PersistentClient(path="./chroma_db")

# 使用免费开箱即用的轻量级 Embedding 函数（默认采用 sentence-transformers/all-MiniLM-L6-v2）
# 如果需要中文优秀支持，可更换为 BAE/bge-small-zh-v1.5
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# --- 2. 创建/获取 Collection (相当于关系型数据库中的 Table) ---
collection = client.get_or_create_collection(
    name="tech_documents",
    embedding_function=sentence_transformer_ef,
    metadata={"hnsw:space": "cosine"} # 指定索引算法使用余弦相似度
)

# --- 3. 准备带有元数据 (Metadata) 的文档数据 ---
# 元数据（如来源、分类、创建日期）是实现【结构化过滤 + 向量检索】的核心！
documents = [
    "ReAct 框架结合了思维链（Thought）与行动（Action），是 Agent 的开山之作。",
    "Agent 工业落地时，如果 Observation 过长会导致显存爆炸（OOM），需要滑动窗口记忆管理。",
    "LoRA 是一种高效微调技术，通过低秩分解大幅减少需要训练的参数量。",
    "DPO 是一种免奖励模型的直接偏好优化对齐算法，比 PPO 更稳定易训练。",
]

metadatas = [
    {"category": "Agent", "author": "Alice", "year": 2026},
    {"category": "Agent", "author": "Bob", "year": 2026},
    {"category": "Fine-Tuning", "author": "Alice", "year": 2025},
    {"category": "Alignment", "author": "Charlie", "year": 2025},
]

ids = ["doc_1", "doc_2", "doc_3", "doc_4"]

# --- 4. 批量写入向量数据库 (Indexing) ---
print("📦 正在写入文档并自动生成 Embedding 与 HNSW 索引...")
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)
print(f"✅ 成功写入 {collection.count()} 条数据！")


# --- 5. 执行结合 Metadata 过滤的高级向量检索 ---
def search_knowledge_base(query: str, category_filter: str = None):
    print(f"\n🔍 [查询]: '{query}' | [分类过滤]: {category_filter}")

    # 构造元数据过滤条件
    where_condition = None
    if category_filter:
        where_condition = {"category": category_filter}

    results = collection.query(
        query_texts=[query],
        n_results=2, # 取 Top-2 相似结果
        where=where_condition # 结合标量过滤（Metadata Filtering）
    )

    # 打印检索结果
    for idx in range(len(results['ids'][0])):
        doc_id = results['ids'][0][idx]
        doc_text = results['documents'][0][idx]
        metadata = results['metadatas'][0][idx]
        distance = results['distances'][0][idx] # 相似度距离 (余弦距离越小越相似)

        print(f"  Top {idx+1} [ID: {doc_id} | 距离: {distance:.4f}]")
        print(f"  📄 内容: {doc_text}")
        print(f"  🏷️ 元数据: {metadata}\n")

if __name__ == "__main__":
    # 1. 纯语义向量检索
    search_knowledge_base(query="Agent 显存爆了怎么办？")

    # 2. 结合分类过滤的向量检索（即使 Fine-Tuning 库里有相似内容，也会被硬性过滤）
    search_knowledge_base(query="如何优化参数减少训练显存？", category_filter="Agent")

1. 先过滤还是后过滤（Pre-Filtering vs. Post-Filtering）：
在上面的代码中，我们结合了 `category_filter`（元数据过滤）与向量检索。
    * 思考：在向量数据库的底层实现中：
        * 后过滤（Post-Filtering）：先按向量相似度召回 Top 100，然后再过滤出 `category == "Agent"` 的文档。这在某些特定分类数据较少时可能会导致什么严重问题（比如最终过滤完结果数量不够 Top K）？
        * 预过滤（Pre-Filtering）：如果在进入 HNSW 图检索之前就提前过滤掉不符合条件的节点，这又会如何破坏 HNSW 图的拓扑连通性？现代向量数据库是如何解决这个两难困境的？

    * 向量数据库增量更新与数据一致性：
        * 试着给上面的 ChromaDB 数据库编写一个 `delete` 和 `update` 脚本（修改 `doc_2` 的内容）。
        * 工程思考：在庞大的 HNSW 图索引中，直接“删除”一个位于图中间关键通道上的向量节点是非常昂贵的（可能导致图断裂）。向量数据库通常采用什么策略来实现高吞吐的删除操作（提示：Soft Delete 软删除与定期 Re-index 标记清理）？

## 双路召回与重排
“检索到了内容，不代表检索到了正确的答案”。在真实的 RAG 生产环境中，只依赖向量数据库的“单路稠密检索（Dense Retrieval）”，往往会遇到严重的精准度瓶颈。这时候需要RAG 召回——**混合检索（Hybrid Search）与重排序（Reranking）**

1. 混合检索的数学互补：理解为什么“向量语义检索（Dense）”和“关键词检索（Sparse / BM25）”必须结合，并掌握 RRF（Reciprocal Rank Fusion，倒数排名融合） 的融合算法。
2. 重排序模型（Reranker / Cross-Encoder）：掌握双编码器（Bi-Encoder）与交叉编码器（Cross-Encoder）的计算差异，手写一个 Vector + BM25 + Rerank 的三阶段高召回 RAG 流水线。

#### 单路检索的痛点与重排的魔法
1. 为什么“纯向量检索”在工业界屡屡翻车？

    向量 Embedding（Dense Retrieval）擅长理解模糊语义和近义词，但它有两个致命弱点：
   * 对专有名词、型号、代码极其不敏感：比如搜索“产品型号 `RTX-4090-Ti`” 或“错误码 `ERR_9021`”，向量模型可能会匹配回一堆关于“显卡”或“报错”的大段通用文字，却偏偏漏掉包含精准型号的特定文档。
   * 数字与精确匹配能力差：对于“2026年财务报表”，向量很难把“2026”这个精确数字的权重提得足够高。
      * 因此，工业界必须引入经典的 BM25（Sparse Retrieval，稀疏检索）。BM25 擅长精准关键词匹配。“向量抓语义，BM25 抓精确词”，两者结合就是混合检索（Hybrid Search）。

2. 双路召回的分数对齐：RRF (Reciprocal Rank Fusion)

    向量检索输出的是余弦距离/点积（如 0.82），而 BM25 输出的是基于词频无上界的得分（如 12.4）。不同打分体系无法直接相加！
   为了合并两路结果，工业界最常用的算法是 RRF（倒数排名融合）。它只看排名，不看原始分数：
   $$RRF\_Score(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
   * $M$：检索路数（这里是 2，即 Vector 路和 BM25 路）。
   * $r_m(d)$：文档 $d$ 在第 $m$ 路检索中的排名位置（从 1 开始）。
   * $k$：常数平滑因子（工业界通常设为 $60$）
   > 原理：不管得分多夸张，只要你在某一路排第 1，你就获得 $\frac{1}{60+1}$ 的得分。如果在两路都排名前列，最终的总分就会很高。

3. Bi-Encoder vs. Cross-Encoder (Reranker)

    有了混合检索召回的 Top 20/50 个文档后，我们需要用 Reranker（重排模型） 精选出 Top 3/5 送给 LLM。
   |特性   | Bi-Encoder ( Embedding 模型 )   |Cross-Encoder ( Reranker 模型 ) |
   |-------|-------------|-------------|
   |计算架构|Query 和 Doc 分别过 Transformer，独立生成向量|Query 和 Doc 拼接在一起 [CLS] Query [SEP] Doc 一起过 Transformer|
   |注意力机制|Query 和 Doc 之间没有交叉注意力 (No Cross-Attention)|Query 的每一个词和 Doc 的每一个词做全量 Cross-Attention|
   |速度与精度|速度极快（可预计算向量预存），精度中等|计算极其昂贵（无法预计算），语义匹配精度极高|
   |定位|海量数据召回阶段 (First-stage Retrieval)|少数候选集的精排阶段 (Second-stage Reranking)|

#### Hybrid Search + RRF + Rerank 流水线
下面的代码模拟了一个完整的工业级召回与重排流程（零重型依赖，仅需 `rank_bm25` 和 `numpy`）：
> 安装依赖：pip install rank-bm25 numpy


In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from typing import List, Dict, Tuple

# --- 1. 模拟数据源 ---
documents = [
    "大模型 Agent 工业落地三大隐形炸弹：结构化输出脆弱、Observation 暴涨、规划死循环。",
    "RAG 系统的核心优化：使用 BM25 结合向量检索构成混合检索 (Hybrid Search)。",
    "用户搜索报错码 ERR_9021 时，纯向量检索往往无法精准命中，需要 BM25 补位。",
    "ReAct 框架结合了 Thought 与 Action，是智能体推理的核心控制流程。",
    "2026年企业级 RAG 架构标准：Vector DB + BM25 + Cross-Encoder Reranker。"
]

# 分词处理 (简化版，实际中文建议使用 jieba)
tokenized_docs = [doc.split(" ") for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# --- 2. 模拟两路召回 (Dense & Sparse) ---
def mock_vector_search(query: str, top_k: int = 4) -> List[Tuple[int, float]]:
    """模拟向量检索：语义接近但对 ERR_9021 不敏感"""
    # 模拟排名：[Doc_ID, Score]
    if "ERR_9021" in query:
        # 向量模型误以为 ERR_9021 与 Agent/ReAct 相关的通用报错最像
        return [(0, 0.88), (1, 0.75), (2, 0.62), (4, 0.55)]
    return [(1, 0.90), (4, 0.85), (0, 0.70), (3, 0.60)]

def bm25_search(query: str, top_k: int = 4) -> List[Tuple[int, float]]:
    """BM25 检索：对精确词 ERR_9021 绝对敏感"""
    tokenized_query = query.split(" ")
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, float(scores[idx])) for idx in top_indices if scores[idx] > 0]

# --- 3. RRF (倒数排名融合) 算法实现 ---
def rrf_fusion(dense_results: List[Tuple[int, float]],
               sparse_results: List[Tuple[int, float]],
               k: int = 60) -> List[Tuple[int, float]]:
    """根据排名而非绝对分数计算 RRF 得分"""
    rrf_scores: Dict[int, float] = {}

    # 累加向量检索的 RRF 分数
    for rank, (doc_id, _) in enumerate(dense_results):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + (rank + 1)))

    # 累加 BM25 检索的 RRF 分数
    for rank, (doc_id, _) in enumerate(sparse_results):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k + (rank + 1)))

    # 按 RRF 得分降序排列
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_rrf

# --- 4. 模拟 Cross-Encoder (Reranker) 打分 ---
def mock_cross_encoder_rerank(query: str, candidate_doc_ids: List[int]) -> List[Tuple[int, float]]:
    """
    模拟 Cross-Encoder：将 Query 与 Candidate 文档全量拼接计算相关度。
    对 Query 与 Doc 的细粒度交互具有绝对精准的感知力。
    """
    rerank_scores = []
    for doc_id in candidate_doc_ids:
        doc_text = documents[doc_id]
        score = 0.1
        # 模拟 Cross-Encoder 的精细语法/逻辑对齐感知
        if "ERR_9021" in query and "ERR_9021" in doc_text:
            score += 0.85  # 准确识别出精确匹配的核心相关度
        if "混合检索" in query and "BM25" in doc_text:
            score += 0.70
        rerank_scores.append((doc_id, score))

    return sorted(rerank_scores, key=lambda x: x[1], reverse=True)


# --- 5. 执行完整三阶段 RAG 召回与重排流水线 ---
if __name__ == "__main__":
    test_query = "ERR_9021 报错 怎么 处理"
    print(f"🔎 【用户查询】: '{test_query}'\n")

    # 阶段 1: 双路召回 (Dual-Path Retrieval)
    dense_res = mock_vector_search(test_query)
    sparse_res = bm25_search(test_query)

    print("📍 [路 1 - 向量召回 Top 3]:", [f"Doc_{idx}" for idx, _ in dense_res[:3]])
    print("📍 [路 2 - BM25 召回 Top 3]:", [f"Doc_{idx}" for idx, _ in sparse_res[:3]])

    # 阶段 2: RRF 排名融合 (Hybrid Fusion)
    fused_res = rrf_fusion(dense_res, sparse_res)
    fused_doc_ids = [doc_id for doc_id, _ in fused_res]
    print(f"\n🔀 [RRF 融合后的候选列表]: {fused_doc_ids}")

    # 阶段 3: Cross-Encoder 重排序 (Reranking)
    final_ranked = mock_cross_encoder_rerank(test_query, fused_doc_ids)

    print("\n🎯 【最终 Rerank 输出 Top 2】(送到 LLM 的上下文):")
    for rank, (doc_id, score) in enumerate(final_ranked[:2]):
        print(f"  Rank {rank+1} [Doc_{doc_id} | Rerank Score: {score:.2f}]:")
        print(f"  >>> {documents[doc_id]}")

1. Reranker 的延迟与成本权衡（Latency vs. Accuracy）：
    * 在上面的架构中，Cross-Encoder（如 `bge-reranker-large`）的精排效果极好，但其计算延迟远高于向量点积。
    * 工程思考：在千万级文档的企业系统中，如果我们把召回阶段的 Top K 设为 1000，然后把这 1000 个文档全部塞给 Reranker 模型进行重排，系统会发生什么情况？在工业界，如何合理设置 召回数（Retrieval K） 与 重排数（Reranking K） 的比例平衡？

2. 多语言与自定义词库对 BM25 的影响：
    * 代码中使用 `doc.split(" ")` 进行简单分词。但在处理中文或特定领域（如医疗、法律、代码）时，词线并不清晰。
    * 动手实践：尝试思考如果企业内部有大量诸如 `QLoRA`、`DPO`、`vLLM` 等专有名词，在使用结巴分词（Jieba）或中文 BM25 时，应该如何配置自定义词典（User Dict）以防止核心专业词被切碎？